In [2]:
import pandas as pd
import yfinance as yf
import vectorbt as vbt
import talib 
import numpy as np

In [18]:
#AMAZON MACD


df = yf.download(tickers = "AMZN", start= "2015-05-05", end= "2025-05-05")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

close = df["Close"]



macd, signal, hist = talib.MACD(close, fastperiod=24, slowperiod=52, signalperiod=18)

entries = macd >= signal
exits = macd <= signal



backtest = vbt.Portfolio.from_signals(
       close,
       entries = entries,
       exits = exits,
       init_cash = 100000,
       size = 10000.0,
       size_type = "value",
       freq = "1D", 
       fees = 0.001)

backtest.plot().show()
print(backtest.stats())

[*********************100%***********************]  1 of 1 completed


Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                                112571.11745
Total Return [%]                            12.571117
Benchmark Return [%]                       802.110677
Max Gross Exposure [%]                      12.660959
Total Fees Paid                             963.41306
Max Drawdown [%]                             6.445537
Max Drawdown Duration              1115 days 00:00:00
Total Trades                                       48
Total Closed Trades                                47
Total Open Trades                                   1
Open Trade PnL                             111.470368
Win Rate [%]                                40.425532
Best Trade [%]                               29.95515
Worst Trade [%]                            -15.829031
Avg Winning Trade [%]       

In [23]:
#AMAZON MACD, DIFF DATE

df = yf.download(tickers = "AMZN", start= "2015-05-05", end= "2026-05-05")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

close = df["Close"]



macd, signal, hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

entries = macd >= signal
exits = macd <= signal



backtest = vbt.Portfolio.from_signals(
       close,
       entries = entries,
       exits = exits,
       init_cash = 100000,
       size = 10000.0,
       size_type = "value",
       freq = "1D", 
       fees = 0.001)

backtest.plot().show()
print(backtest.stats())


[*********************100%***********************]  1 of 1 completed


Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               107176.085364
Total Return [%]                             7.176085
Benchmark Return [%]                      1191.816003
Max Gross Exposure [%]                      12.206748
Total Fees Paid                           2316.572965
Max Drawdown [%]                             3.990004
Max Drawdown Duration              1394 days 00:00:00
Total Trades                                      116
Total Closed Trades                               115
Total Open Trades                                   1
Open Trade PnL                            2909.693134
Win Rate [%]                                40.869565
Best Trade [%]                              26.329829
Worst Trade [%]                            -14.039824
Avg Winning Trade [%]       

In [20]:
#SNP500 MACD

import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2025-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")


valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns)
)
close_df, high_df, low_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
)

[*************         27%                       ]  137 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[****************      33%                       ]  166 of 500 completed$Q: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[********************  42%                       ]  208 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[**********************64%******                 ]  321 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[*********************100%***********************]  500 of 500 

In [21]:
#WITH 10% STOPLOSS

macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )

entries = macd_df >= signal_df
exits = macd_df < signal_df


backtest5 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
    tp_stop = 0.10
       
)


print(backtest5.stats())
backtest5.plot().show()


Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               233073.455735
Total Return [%]                           133.073456
Benchmark Return [%]                       337.427314
Max Gross Exposure [%]                          100.0
Total Fees Paid                          74542.508278
Max Drawdown [%]                            23.967539
Max Drawdown Duration               416 days 00:00:00
Total Trades                                    58306
Total Closed Trades                             57913
Total Open Trades                                 393
Open Trade PnL                             5741.33716
Win Rate [%]                                40.023829
Best Trade [%]                              63.062095
Worst Trade [%]                              -44.2802
Avg Winning Trade [%]       

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



In [22]:
entries = (macd_df >= signal_df) & (rsi_df >= 30)
exits = (macd_df < signal_df) | (rsi_df > 70)



backtest6 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
    tp_stop = 0.10
       
)


print(backtest6.stats())
backtest6.plot().show()

Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               218555.994901
Total Return [%]                           118.555995
Benchmark Return [%]                       337.427314
Max Gross Exposure [%]                          100.0
Total Fees Paid                          74080.577771
Max Drawdown [%]                            26.167659
Max Drawdown Duration               417 days 00:00:00
Total Trades                                    56016
Total Closed Trades                             55628
Total Open Trades                                 388
Open Trade PnL                            5399.140492
Win Rate [%]                                40.184799
Best Trade [%]                              63.062095
Worst Trade [%]                              -44.2802
Avg Winning Trade [%]       

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



In [28]:
#WITH 10% STOPLOSS

macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )

entries = macd_df >= signal_df
exits = macd_df < signal_df


backtest5 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.20,
       
)


print(backtest5.stats())
backtest5.plot().show()

Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               275012.819658
Total Return [%]                            175.01282
Benchmark Return [%]                       337.427314
Max Gross Exposure [%]                          100.0
Total Fees Paid                          62382.173948
Max Drawdown [%]                            30.490136
Max Drawdown Duration               206 days 00:00:00
Total Trades                                    47466
Total Closed Trades                             47023
Total Open Trades                                 443
Open Trade PnL                           24241.973164
Win Rate [%]                                38.155796
Best Trade [%]                             182.763646
Worst Trade [%]                            -54.112382
Avg Winning Trade [%]       

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data

